In [0]:
# Create Sample Delta Table
empData = [
    (111, "Stephen", "King", 2000),
    (222, "Philipp", "Larkin", 8000),
    (333, "John", "Smith", 6000)
]
empSchema = ["Id", "FirstName", "LastName", "salary"]
df = spark.createDataFrame(data=empData, schema=empSchema)
display(df)


#APPLY array_repeat

In [0]:
#APPLY array_repeat
from pyspark.sql.functions import explode, array_repeat, col
# Create a new column 'key_col' by repeating the 'Id' value 5 times
repeatDf = df.withColumn("key_col", array_repeat(col("Id"), 5))
display(repeatDf)


#Apply explode ,array_repeat

In [0]:
#Apply explode ,array_repeat

from pyspark.sql.functions import explode, array_repeat, col
# Repeat the 'Id' column 10 times and explode it to create multiple rows
df1 = df.withColumn("key_col", explode(array_repeat(col("Id"), 10)))
display(df1)


In [0]:
#Apply window function

from pyspark.sql.window import Window
from pyspark.sql.functions import lit, row_number, col
# Define a window spec that assigns row numbers arbitrarily
win = Window().orderBy(lit('A'))
# Add row number column
df2 = df1.withColumn("RowN", row_number().over(win))
# Create a new ID by adding RowN to Id
df3 = df2.withColumn("NewID", col("Id") + col("RowN"))
display(df3)







Window().orderBy(lit('A')) is used when you need to generate sequential row numbers but do not require any specific ordering of the rows. Since lit('A') assigns the same value to every row, Spark has no meaningful sort column and therefore assigns row numbers based on its internal processing and partitioning. This approach is commonly used to create surrogate keys, unique sequence numbers, or temporary IDs when row order is not important.

In [0]:

#Select related columns
outputDF = df3.drop("Id", "key_col", "RowN") \
              .select(col("NewID").alias("Id"), "FirstName", "LastName", "salary")
display(outputDF)



# #Function definition

In [0]:
#Function definition
def multiplyInputData(df, keyCol, multiplyFactor):
    from pyspark.sql.functions import explode, array_repeat, col, lit, row_number
    from pyspark.sql.window import Window
    # Step 1: Repeat the key column values and explode them into multiple rows
    df1 = df.withColumn(f"{keyCol}", explode(array_repeat(col(f"{keyCol}"), multiplyFactor)))
    # Step 2: Create a window spec to assign row numbers
    win = Window().orderBy(lit('A'))
    df2 = df1.withColumn("RowN", row_number().over(win))
    # Step 3: Update the key column by adding the row number to the original Id
    df3 = df2.withColumn(f"{keyCol}", col("Id") + col("RowN")).drop("RowN")
    return df3


finalDF = multiplyInputData(df, "Id", 10)
display(finalDF)
